# Valence Band Offset of the Si/Ge (001) Strained Superlattice

> **Chris G. Van de Walle and Richard M. Martin**, "Theoretical calculations of heterojunction discontinuities in
> the Si/Ge system", Physical Review B 34, 5621 (1986).
> [DOI:10.1103/PhysRevB.34.5621](https://doi.org/10.1103/PhysRevB.34.5621)

Computes the valence band offset of Ge grown pseudomorphically on Si(001) from the superlattice and the two strained
bulk cells created in the [structure notebook](interface_3d_3d_silicon_germanium.ipynb), with the platform's
[Valence Band Offset](../workflows/valence_band_offset.ipynb) workflow, and compares it with Van de Walle & Martin
(1986). As in the paper (Sec. III.A), each bulk's valence band maximum is measured from its own average potential
V̄, and the superlattice gives the step in V̄ between the two materials:

ΔE_v = (E_VBM − V̄)_Ge − (E_VBM − V̄)_Si + (V̄_Ge − V̄_Si)_superlattice

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](interface_3d_3d_silicon_germanium.ipynb), which saves `Si/Ge (001) superlattice 4+4`, `Si (001) bulk a=5.43` and `Ge (001) bulk a=5.43 c=5.82` to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.4, or use the defaults.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the results; the last section prints the comparison with the manuscript.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the superlattice and the two bulks by name from the `uploads` folder or the platform, print their provenance.
1. Configure the DFT model and a k-grid per material.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Configure the Valence Band Offset workflow: the model, a k-grid per material and the band structures the valence band maxima are read from.
1. Run the job, or reuse one that already ran on the same materials with the same settings.
1. Retrieve the valence band maxima and the potential profiles, compute the offset, and plot the potential with the bulk potentials overlaid, as in the paper's Fig. 2.
1. Compare with Van de Walle & Martin (1986).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")

### 1.2. Material names

In [ ]:
# Names saved by interface_3d_3d_silicon_germanium.ipynb.
SUPERLATTICE_NAME = "Si/Ge (001) superlattice 4+4"
SUBSTRATE_BULK_NAME = "Si (001) bulk a=5.43"
FILM_BULK_NAME = "Ge (001) bulk a=5.43 c=5.82"

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

VBO_WORKFLOW_SEARCH_TERM = "valence_band_offset.json"
MY_WORKFLOW_NAME = "Valence Band Offset"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = "001"  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.OR
PPN = 16  # the OR queue allows at most 16 cores per node
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
MODEL_SUBTYPE = "lda"
FUNCTIONAL = "pz"  # Van de Walle & Martin use LDA (Ceperley-Alder)
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the LDA family the platform publishes for Si and Ge
VALENCE_ELECTRONS_PER_ATOM = {"Si": 4, "Ge": 14}  # GBRV pseudopotentials, Ge with its 3d shell
ECUTWFC = 40   # Ry, GBRV's recommended wavefunction cutoff
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff
KPOINT_DENSITY = 5  # points per Å⁻¹: the same sampling in the superlattice and in the bulks
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KPOINT_DENSITY}"

SCF_UNIT = "pw_scf"
BANDS_UNIT = "pw_bands"
AVERAGE_UNIT = "average ESP"

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load from the uploads folder and print provenance

Si planes are 1.3575 Å apart, Ge planes 1.455 Å and both interfaces 1.406 Å (Sec. II, Table I).

In [ ]:
import numpy as np
from collections import Counter
from mat3ra.notebooks_utils.core.entity.material.api import load_material

MATERIAL_NAMES = [SUPERLATTICE_NAME, SUBSTRATE_BULK_NAME, FILM_BULK_NAME]
materials_by_name = {name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in MATERIAL_NAMES}


def get_plane_spacings(material):
    cartesian_material = material.clone()
    cartesian_material.to_cartesian()
    heights = np.array(cartesian_material.basis.coordinates.values)[:, 2]
    plane_heights = np.unique(np.round(heights, 4) % round(material.lattice.c, 4))
    return np.diff(np.append(plane_heights, plane_heights[0] + material.lattice.c))


for name, material in materials_by_name.items():
    composition = Counter(material.basis.elements.values)
    electron_count = sum(VALENCE_ELECTRONS_PER_ATOM[element] * count for element, count in composition.items())
    lattice = material.lattice
    print(f"{name}: {dict(composition)}, {material.basis.number_of_atoms} atoms, "
          f"cell {lattice.a:.4f} x {lattice.b:.4f} x {lattice.c:.4f} Å, {electron_count} valence electrons")
    print(f"  plane spacings along c: {np.round(get_plane_spacings(material), 4).tolist()} Å")

## 4. Configure the DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

### 4.2. k-grid per material

In [ ]:
from mat3ra.notebooks_utils.workflow import kgrid_from_density

kgrid = {name: kgrid_from_density(material, KPOINT_DENSITY) for name, material in materials_by_name.items()}
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Configure the Valence Band Offset workflow

The workflow runs the same band structure and electrostatic potential calculation on the superlattice and on the
two bulks, each with its own k-grid, and stores the band structures. Section 8 computes the offset from them and
from the planar-averaged potential, as the paper does, so the workflow's own lineup step is left out.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.wode.context.providers import PointsGridDataProvider
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

superlattice = materials_by_name[SUPERLATTICE_NAME]
job_materials = [superlattice, materials_by_name[SUBSTRATE_BULK_NAME], materials_by_name[FILM_BULK_NAME]]

workflow = Workflow.create(
    WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(VBO_WORKFLOW_SEARCH_TERM)
)
workflow.name = f"{MY_WORKFLOW_NAME} {' | '.join(MATERIAL_NAMES)} {MODEL_TAG}"
for subworkflow in list(workflow.subworkflows):
    if subworkflow.application.name != APPLICATION_NAME:
        workflow.remove_unit(workflow.get_unit_by_name(name=subworkflow.name).flowchartId)
        workflow.remove_subworkflow_by_id(subworkflow.id)
for unit_name in [SCF_UNIT, BANDS_UNIT]:
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=unit_name)
for subworkflow in workflow.subworkflows:
    subworkflow.model = model
    material_index = int(next(unit.value for unit in subworkflow.units if unit.name.startswith("Set Material Index")))
    material = job_materials[material_index]

    bands_unit = subworkflow.get_unit_by_name(name=BANDS_UNIT)
    bands_unit.results.append({"name": "band_structure"})
    subworkflow.set_unit(bands_unit)

    scf_unit = subworkflow.get_unit_by_name(name=SCF_UNIT)
    scf_unit.add_context(PointsGridDataProvider(material=material, dimensions=kgrid[MATERIAL_NAMES[material_index]],
                                                isEdited=True).get_context_item_data())
    subworkflow.set_unit(scf_unit)

visualize_workflow(workflow)

## 7. Save the materials and run the job

A job that already ran on the same three materials with the same settings is reused; otherwise a new one is created
and submitted.

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

saved_materials = [get_or_create_material(client, material, ACCOUNT_ID) for material in job_materials]

material_ids = [material["_id"] for material in saved_materials]
candidates = client.jobs.list({
    "_material._id": material_ids[0], "owner._id": ACCOUNT_ID, "workflow.name": workflow.name,
    "status": {"$in": ["submitted", "queued", "active", "finished"]},
})
job = next((candidate for candidate in candidates
            if [material["_id"] for material in candidate["_materials"]] == material_ids), None)
if job is None:
    job = create_job(api_client=client, materials=saved_materials, workflow=workflow, project_id=project_id,
                     owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}")
    submit_jobs(client.jobs, [job["_id"]])
    print(f"✅ Submitted job {job['_id']}")
else:
    print(f"♻️  Reusing job {job['_id']}")
await wait_for_jobs_to_finish_async(client.jobs, [job["_id"]], poll_interval=POLL_INTERVAL)
job_id = job["_id"]
job_status = client.jobs.get(job_id)["status"]
if job_status != "finished":
    raise RuntimeError(f"Job {job_id} ended with status '{job_status}'; "
                       "its page on the platform shows the failed unit.")

## 8. Retrieve the results
### 8.1. Valence band maxima and average potentials

E_VBM is the top of the highest valence band, band N_electrons / 2, along each bulk's k-path, and the gap is the
bottom of the next band minus E_VBM. In the LDA the strained Ge is a semimetal: its L conduction band dips below the
valence band top at Γ, so its gap comes out negative. V̄ is the planar average of the electrostatic
potential, averaged over the whole cell for a bulk, and over one plane spacing at the centre of each block for the
superlattice, the bulk-like region one plane away from both interfaces (Fig. 2).

In [ ]:
band_structures, profiles = {}, {}
for subworkflow in client.jobs.get(job_id)["workflow"]["subworkflows"]:
    units = {unit["name"]: unit for unit in subworkflow["units"]}
    name = MATERIAL_NAMES[int(next(unit["value"] for unit in subworkflow["units"]
                                   if unit["name"].startswith("Set Material Index")))]
    band_structures[name] = np.array(client.properties.get_for_job(
        job_id, property_name="band_structure", unit_id=units[BANDS_UNIT]["flowchartId"])[0]["yDataSeries"])
    profiles[name] = client.properties.get_for_job(
        job_id, property_name="average_potential_profile", unit_id=units[AVERAGE_UNIT]["flowchartId"])[0]


superlattice_in_cartesian = superlattice.clone()
superlattice_in_cartesian.to_cartesian()
heights = np.array(superlattice_in_cartesian.basis.coordinates.values)[:, 2]
elements = np.array(superlattice.basis.elements.values)
period = superlattice.lattice.c
positions = np.array(profiles[SUPERLATTICE_NAME]["xDataArray"])
planar_average = np.array(profiles[SUPERLATTICE_NAME]["yDataSeries"][0])
block_windows, superlattice_potential, bulk_potential, valence_band_maximum = {}, {}, {}, {}
for bulk_name in (SUBSTRATE_BULK_NAME, FILM_BULK_NAME):
    bulk = materials_by_name[bulk_name]
    element = bulk.basis.elements.values[0]
    number_of_valence_bands = VALENCE_ELECTRONS_PER_ATOM[element] * bulk.basis.number_of_atoms // 2
    # centre of the block's planes, also when the block wraps around the cell boundary
    center = np.angle(np.exp(2j * np.pi * heights[elements == element] / period).mean()) * period / (2 * np.pi) % period
    spacing = get_plane_spacings(bulk).mean()
    block_windows[bulk_name] = (center, spacing)
    distances = (positions - center + period / 2) % period - period / 2
    superlattice_potential[bulk_name] = planar_average[np.abs(distances) <= spacing / 2].mean()
    bulk_potential[bulk_name] = np.mean(profiles[bulk_name]["yDataSeries"][0])
    valence_band_maximum[bulk_name] = band_structures[bulk_name][number_of_valence_bands - 1].max()
    gap = band_structures[bulk_name][number_of_valence_bands].min() - valence_band_maximum[bulk_name]
    print(f"{bulk_name}: E_VBM = {valence_band_maximum[bulk_name]:.4f} eV, V̄ = {bulk_potential[bulk_name]:.4f} eV, "
          f"E_VBM - V̄ = {valence_band_maximum[bulk_name] - bulk_potential[bulk_name]:.4f} eV, "
          f"gap {gap:+.3f} eV")
    print(f"  in the superlattice: V̄ = {superlattice_potential[bulk_name]:.4f} eV, averaged over "
          f"{spacing:.4f} Å centred at z = {center:.4f} Å")

### 8.2. Valence band offset

In [ ]:
bulk_term = ((valence_band_maximum[FILM_BULK_NAME] - bulk_potential[FILM_BULK_NAME])
             - (valence_band_maximum[SUBSTRATE_BULK_NAME] - bulk_potential[SUBSTRATE_BULK_NAME]))
potential_lineup = superlattice_potential[FILM_BULK_NAME] - superlattice_potential[SUBSTRATE_BULK_NAME]
valence_band_offset = bulk_term + potential_lineup
print(f"(E_VBM - V̄)_Ge - (E_VBM - V̄)_Si = {bulk_term:+.4f} eV")
print(f"(V̄_Ge - V̄_Si)_superlattice     = {potential_lineup:+.4f} eV")
print(f"ΔE_v = E_VBM(Ge) - E_VBM(Si)     = {valence_band_offset:+.4f} eV")

### 8.3. Potential across the superlattice

Drawn as Fig. 2 of the paper: one period from the Ge→Si interface, Si on the left. Each bulk's own potential
(dotted) is overlaid on its block, shifted to the block's average; where they coincide the block is bulk-like, and the
two averages (dashed horizontal lines) are shifted with respect to each other by the lineup.

In [ ]:
import matplotlib.pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

silicon_center, silicon_spacing = block_windows[SUBSTRATE_BULK_NAME]
# one atom per (001) plane in the primitive cell, so atoms count planes
silicon_planes = np.sum(elements == materials_by_name[SUBSTRATE_BULK_NAME].basis.elements.values[0])
plot_origin = silicon_center - silicon_planes * silicon_spacing / 2  # the middle of the Ge->Si interface
order = np.argsort((positions - plot_origin) % period)

figure = plt.figure(figsize=(10, 5))
plt.plot(((positions - plot_origin) % period)[order], planar_average[order], color="black", label="superlattice")
for bulk_name, color in ((SUBSTRATE_BULK_NAME, "tab:blue"), (FILM_BULK_NAME, "tab:red")):
    bulk = materials_by_name[bulk_name]
    element = bulk.basis.elements.values[0]
    center, spacing = block_windows[bulk_name]
    half_width = np.sum(elements == element) * spacing / 2
    offsets = np.linspace(-half_width, half_width, 400)  # from the block's centre
    plane_offset = (heights[elements == element][0] - center + period / 2) % period - period / 2
    bulk_profile = profiles[bulk_name]
    bulk_in_cartesian = bulk.clone()
    bulk_in_cartesian.to_cartesian()
    bulk_plane = bulk_in_cartesian.basis.coordinates.values[0][2]
    bulk_values = np.interp(offsets - plane_offset + bulk_plane,
                            bulk_profile["xDataArray"], bulk_profile["yDataSeries"][0], period=bulk.lattice.c)
    region = offsets + (center - plot_origin) % period
    shift = superlattice_potential[bulk_name] - bulk_potential[bulk_name]
    plt.plot(region, bulk_values + shift, color=color, linestyle=":", linewidth=2, label=f"bulk {element}, shifted")
    plt.hlines(superlattice_potential[bulk_name], region[0], region[-1], color=color, linestyle="--",
               label=f"V̄_{element} = {superlattice_potential[bulk_name]:.3f} eV")
for height, element in zip(heights, elements):
    plt.text((height - plot_origin) % period, 0.02, element, transform=plt.gca().get_xaxis_transform(), ha="center")
plt.xlim(0, period)
plt.xlabel("position along (001) (Å)")
plt.ylabel("V̄(z), electrostatic (eV)")
plt.title(f"{SUPERLATTICE_NAME}: ΔV̄ = {potential_lineup:.3f} eV, ΔE_v = {valence_band_offset:.3f} eV")
plt.legend(loc="lower right", bbox_to_anchor=(1, 0.07), fontsize=8)
display_matplotlib_figure(figure)

## 9. Compare with Van de Walle & Martin (1986)

Settings that differ from the manuscript: GBRV ultrasoft LDA pseudopotentials at 40/200 Ry and a k-grid of 5 points
per Å⁻¹, against norm-conserving pseudopotentials at 6 Ry and four special points (Sec. III.A); V̄ here is the
electrostatic potential, while the paper uses the l = 1 component of the total potential. The two terms of the offset
depend on that reference, only their sum does not (p. 5625). Spin–orbit coupling is in neither calculation: the
paper adds 0.10 eV, a third of germanium's experimental splitting, to get Table I's 0.84 eV (Sec. IV).

In [ ]:
VAN_DE_WALLE_OFFSET = 0.74  # eV, Sec. III.A, no spin-orbit
SPIN_ORBIT_SHIFT = 0.10  # eV, raises the Ge valence band top (Sec. IV)
VAN_DE_WALLE_OFFSET_WITH_SPIN_ORBIT = 0.84  # eV, Table I, (001), a∥ = 5.43 Å
VAN_DE_WALLE_BULK_TERM = 11.08 - 11.19  # eV, E_VBM - V̄ of strained Ge and of Si, l = 1 reference (p. 5625)
VAN_DE_WALLE_LINEUP = 0.85  # eV, ΔV̄, l = 1 reference (p. 5625)

offset_with_spin_orbit = valence_band_offset + SPIN_ORBIT_SHIFT
print(f"ΔE_v = {valence_band_offset:.3f} eV   paper {VAN_DE_WALLE_OFFSET:.2f} eV   "
      f"deviation {100 * (valence_band_offset / VAN_DE_WALLE_OFFSET - 1):+.1f} %")
print(f"ΔE_v + {SPIN_ORBIT_SHIFT:.2f} eV = {offset_with_spin_orbit:.3f} eV   "
      f"paper {VAN_DE_WALLE_OFFSET_WITH_SPIN_ORBIT:.2f} eV   deviation "
      f"{100 * (offset_with_spin_orbit / VAN_DE_WALLE_OFFSET_WITH_SPIN_ORBIT - 1):+.1f} %")
print(f"Ge valence band top {'above' if valence_band_offset > 0 else 'below'} Si's   paper above")
print(f"(E_VBM - V̄)_Ge - (E_VBM - V̄)_Si = {bulk_term:+.2f} eV   paper {VAN_DE_WALLE_BULK_TERM:+.2f} eV")
print(f"(V̄_Ge - V̄_Si)_superlattice = {potential_lineup:+.2f} eV   paper {VAN_DE_WALLE_LINEUP:+.2f} eV")

## References

[1] C. G. Van de Walle and R. M. Martin, "Theoretical calculations of heterojunction discontinuities in the Si/Ge
system", Phys. Rev. B 34, 5621 (1986). [DOI: 10.1103/PhysRevB.34.5621](https://doi.org/10.1103/PhysRevB.34.5621)

[2] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT
calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053